In [1]:
import pandas as pd

# BPR Hyperparameter tuning

In [2]:
from dynamicTasteDistortion.ioUtils import get_cv_results_path

from dynamicTasteDistortion.simulationConstants import input_size_to_file_name


import numpy as np
import pandas as pd

from dynamicTasteDistortion.simulationConstants import (
    USER_COL,
    ITEM_COL,
)

from dynamicTasteDistortion.simulationConstants import input_size_to_file_name

from bprMf.bpr_mf import bprMFWithClickDebiasing, bprMf

import torch
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
size = "s"
num_users = 1000
file_size = input_size_to_file_name[size]

In [4]:
from dynamicTasteDistortion.simulationConstants import MOVIELENS_PATH, YELP_PATH, FOOD_PATH


DATA_TO_PATH = {"ml": MOVIELENS_PATH, "yelp": YELP_PATH, "food": FOOD_PATH}

In [5]:
from bprMf.utils.data import temporal_train_val_test_split
import random


## Movielens

In [6]:
def yield_model(data_type, file_size, num_users):
    results_path = get_cv_results_path(
        data_type, file_size=file_size, num_users=num_users, model_type="bpr"
    )
    file_base_path = DATA_TO_PATH[data_type]
    file_path = f"{file_base_path}/{data_type}_{file_size}.pkl"
    print(f"Loading base dataset from {file_path}...")
    df = pd.read_pickle(file_path)

    results = pd.read_csv(results_path).drop(columns=["Unnamed: 0"])
    winning_model_params = (
        results.sort_values(by="map", ascending=False)
        .iloc[0][["factors", "lr", "reg_lambda", "num_negatives", "n_epochs"]]
        .to_dict()
    )

    int_params = ["factors", "num_negatives", "n_epochs"]
    winning_model_params = {
        k: int(v) if k in int_params else v
        for k, v in winning_model_params.items()
    }

    print(f"Best params for {data_type}_{file_size}: {winning_model_params}")

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    n_users = df.user.max() + 1
    n_items = df.item.max() + 1


    model = bprMf(
                    num_users=n_users,
                    num_items=n_items,
                    dev=dev,
                    **winning_model_params,
                )

    return model, df




def sample_eval_candidates(oracle_df_pos, train_df, n_items, n_negatives=99):
    train_pos = train_df.groupby("user")["item"].apply(set)
    oracle_pos = oracle_df_pos.groupby("user")["item"].apply(set)
    all_items = set(range(n_items))

    rows = []
    for user_id, pos_items in oracle_pos.items():
        seen_items = train_pos.get(user_id, set())
        candidates = list(all_items - seen_items - pos_items)
        neg_samples = random.sample(candidates, min(n_negatives, len(candidates)))
        for item in list(pos_items) + neg_samples:
            rows.append({"user": user_id, "item": item})

    return pd.DataFrame(rows)


def fit_evaluate(model, df, k=5):
    train_df, val_df, test_df = temporal_train_val_test_split(
        df=df,
        user_col=USER_COL,
        val_pct=0.0,
        test_pct=0.3,
    )

    test_df = pd.concat([val_df, test_df])
    pos_truth_set = test_df[test_df["binarized_rating"] == 1].copy()

    model.fit(train_df)
    return model.evaluate(
        train_df=train_df, oot_df=test_df, oracle_df_pos=pos_truth_set, k=k
    )

def fit_evaluate_with_random_negative(model, df, k=5):
    train_df, val_df, test_df = temporal_train_val_test_split(
        df=df,
        user_col=USER_COL,
        val_pct=0.0,
        test_pct=0.3,
    )
    n_items = df["item"].max() + 1
    test_df = pd.concat([val_df, test_df])
    pos_truth_set = test_df[test_df["binarized_rating"] == 1].copy()
    sampled_oracle = sample_eval_candidates(pos_truth_set, train_df, n_items)

    model.fit(train_df)
    return model.evaluate(
        train_df=train_df, oot_df=test_df, oracle_df_pos=sampled_oracle, k=k
    )
    


In [7]:
ml_results_path = get_cv_results_path(
    "ml", file_size=file_size, num_users=num_users, model_type="bpr"
)
ml_results = pd.read_csv(ml_results_path).drop(columns=["Unnamed: 0"])
ml_results

,factors,lr,reg_lambda,num_negatives,n_epochs,map
0,64,0.00010,0.00001,10,10,0.046987
1,64,0.00001,0.00010,10,10,0.045919
2,16,0.01000,0.01000,1,7,0.045562
3,128,0.00100,0.00010,10,7,0.045265
4,64,0.00010,0.01000,5,7,0.044231
5,16,0.01000,0.00100,5,7,0.043824
6,128,0.00010,0.00001,10,7,0.043413
7,32,0.01000,0.01000,5,10,0.043300
8,128,0.00010,0.01000,10,10,0.042472
9,32,0.00010,0.00001,5,10,0.041872


In [8]:
ml_model, ml_1m = yield_model("ml", file_size, num_users)

Loading base dataset from dynamicTasteDistortion/data/movielens/ml_1m.pkl...


Best params for ml_1m: {'factors': 64, 'lr': 0.0001, 'reg_lambda': 1e-05, 'num_negatives': 10, 'n_epochs': 10}


In [9]:
for k in [5, 10, 20]:
    map_movielens = fit_evaluate(ml_model, ml_1m, k)
    print(f"Map@{k}: {map_movielens}")

Epochs: 100%|██████████| 10/10 [06:16<00:00, 37.63s/it]


Map@5: 0.10104928909952608


Epochs: 100%|██████████| 10/10 [06:20<00:00, 38.01s/it]


Map@10: 0.07904509844661052


Epochs: 100%|██████████| 10/10 [06:28<00:00, 38.88s/it]


Map@20: 0.06417352492912355


In [95]:
map_movielens

0.10162496050552923

## Yelp

In [13]:
yelp_results_path = get_cv_results_path(
    "yelp", file_size=file_size, num_users=num_users, model_type="bpr"
)
yelp_results = pd.read_csv(yelp_results_path).drop(columns=["Unnamed: 0"])
yelp_results

,factors,lr,reg_lambda,num_negatives,n_epochs,map
0,32,0.00010,0.00001,5,10,0.049794
1,16,0.01000,0.01000,1,7,0.047840
2,16,0.01000,0.00100,5,7,0.046808
3,128,0.00100,0.00001,10,7,0.046670
4,32,0.01000,0.01000,5,10,0.046347
5,64,0.00001,0.00010,10,10,0.045944
6,64,0.00010,0.00001,10,10,0.045501
7,64,0.01000,0.00100,10,10,0.045057
8,128,0.00010,0.00001,10,7,0.043935
9,64,0.00010,0.01000,5,7,0.043688


In [24]:
yelp_model, yelp_1m = yield_model("yelp", file_size, num_users)

Loading base dataset from dynamicTasteDistortion/data/yelp/yelp_1m.pkl...
Best params for yelp_1m: {'factors': 32, 'lr': 0.0001, 'reg_lambda': 1e-05, 'num_negatives': 5, 'n_epochs': 10}


In [26]:
yelp_1m

,user,item,rating,date,genres,binarized_rating,timestamp
0,107802,13009,3.0,2018-07-07 22:09:11,"[restaurants, breakfast-&-brunch, food, juice-...",0,1531001351
1,20771,13545,3.0,2014-02-05 20:30:30,"[restaurants, breakfast-&-brunch]",0,1391632230
2,84024,15924,4.0,2017-01-14 20:54:15,"[sandwiches, beer, wine-&-spirits, bars, food,...",1,1484427255
3,117237,16955,5.0,2015-01-03 23:21:18,"[supernatural-readings, tours, hotels-&-travel...",1,1420327278
4,5512,22176,5.0,2015-06-21 14:48:06,"[shopping, jewelry]",1,1434898086
...,...,...,...,...,...,...,...
465282,61573,18891,1.0,2018-04-21 22:16:51,"[hotels, hotels-&-travel, event-planning-&-ser...",0,1524349011
465283,38791,16597,3.0,2011-11-13 21:35:09,"[food, specialty-food, korean, chinese, ethnic...",0,1321220109
465284,97879,16139,5.0,2015-02-22 22:53:08,"[arts-&-entertainment, music-venues, nightlife...",1,1424645588
465285,111780,17947,3.0,2017-11-15 09:43:07,"[ice-cream-&-frozen-yogurt, food]",0,1510738987


In [70]:
yelp_filtered = yelp_1m[yelp_1m.groupby("user")["item"].transform("count") >= 20]
print(f"Users: {yelp_filtered['user'].nunique()}")
print(f"Items: {yelp_filtered['item'].nunique()}")
print(f"Interactions: {len(yelp_filtered)}")

Users: 2728
Items: 16223
Interactions: 97480


In [71]:
map_yelp = fit_evaluate_with_random_negative(yelp_model, yelp_filtered)

Epochs: 100%|██████████| 10/10 [00:21<00:00,  2.13s/it]


In [73]:
map_yelp

0.022732059759980407

In [58]:
map_yelp

0.0206856682178137

# Oracle model

## [ML] Oracle model

In [18]:
path = "dynamicTasteDistortion/artifacts/results/ml_1m/oracle_model_f1_results.pkl"

In [19]:
results = pd.read_pickle(path)

In [20]:
results

,model_name,params,validation_f1,test_f1,is_winning_variant
0,SVD,"{'n_factors': 15, 'n_epochs': 30, 'lr_all': 0....",0.505645,NaN,False
1,SVD,"{'n_factors': 30, 'n_epochs': 30, 'lr_all': 0....",0.510479,NaN,False
2,SVD,"{'n_factors': 15, 'n_epochs': 10, 'lr_all': 0....",0.477281,NaN,False
3,SVD,"{'n_factors': 100, 'n_epochs': 30, 'lr_all': 0...",0.503008,NaN,False
4,SVD,"{'n_factors': 30, 'n_epochs': 10, 'lr_all': 0....",0.500629,NaN,False
...,...,...,...,...,...
80,NMF,"{'n_factors': 30, 'n_epochs': 100, 'reg_pu': 0...",0.560511,NaN,False
81,NMF,"{'n_factors': 100, 'n_epochs': 100, 'reg_pu': ...",0.647412,NaN,False
82,NMF,"{'n_factors': 100, 'n_epochs': 50, 'reg_pu': 0...",0.721591,NaN,False
83,NMF,"{'n_factors': 100, 'n_epochs': 100, 'reg_pu': ...",0.753906,NaN,False


In [21]:
winners = results[results["is_winning_variant"] == True]

In [22]:
pd.set_option('display.max_colwidth', None)

In [23]:
winners

,model_name,params,validation_f1,test_f1,is_winning_variant
16,SVD,"{'n_factors': 30, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.02}",0.575029,0.560252,True
45,SVD++,"{'n_factors': 30, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.02}",0.573192,0.562719,True
84,NMF,"{'n_factors': 100, 'n_epochs': 50, 'reg_pu': 0.05, 'reg_qi': 0.05}",0.757246,0.747022,True


## Yelp 1m

In [27]:
yelp = pd.read_pickle("dynamicTasteDistortion/artifacts/results/yelp_1m/oracle_model_f1_results.pkl")

In [28]:
yelp

,model_name,params,validation_f1,test_f1,is_winning_variant
0,SVD,"{'n_factors': 15, 'n_epochs': 30, 'lr_all': 0.002, 'reg_all': 0.05}",0.651385,NaN,False
1,SVD,"{'n_factors': 30, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}",0.676120,0.683712,True
2,SVD,"{'n_factors': 15, 'n_epochs': 10, 'lr_all': 0.002, 'reg_all': 0.1}",0.579866,NaN,False
3,SVD,"{'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}",0.666993,NaN,False
4,SVD,"{'n_factors': 30, 'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.05}",0.643885,NaN,False
...,...,...,...,...,...
80,NMF,"{'n_factors': 30, 'n_epochs': 100, 'reg_pu': 0.05, 'reg_qi': 0.05}",0.539459,NaN,False
81,NMF,"{'n_factors': 100, 'n_epochs': 100, 'reg_pu': 0.02, 'reg_qi': 0.1}",0.633116,NaN,False
82,NMF,"{'n_factors': 100, 'n_epochs': 50, 'reg_pu': 0.02, 'reg_qi': 0.05}",0.817009,0.816323,True
83,NMF,"{'n_factors': 100, 'n_epochs': 100, 'reg_pu': 0.02, 'reg_qi': 0.02}",0.801510,NaN,False


In [29]:
yelp[yelp["is_winning_variant"]]

,model_name,params,validation_f1,test_f1,is_winning_variant
1,SVD,"{'n_factors': 30, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}",0.676120,0.683712,True
30,SVD++,"{'n_factors': 30, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}",0.673943,0.679041,True
82,NMF,"{'n_factors': 100, 'n_epochs': 50, 'reg_pu': 0.02, 'reg_qi': 0.05}",0.817009,0.816323,True
